# Latency Profiling — ONNX FP32 vs INT8

Measures:
- Mean and p95 inference latency per format
- Warm-up effect (first N runs are slower)
- Throughput (requests/sec)

Run N=200 iterations, discard first 10 as warm-up.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd

In [ ]:
N_RUNS = 200
WARMUP = 10
IMGSZ = 640
dummy = np.random.rand(1, 3, IMGSZ, IMGSZ).astype(np.float32)


def benchmark(model_path: str, n: int = N_RUNS, warmup: int = WARMUP) -> list[float]:
    sess = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    inp_name = sess.get_inputs()[0].name
    latencies = []
    for i in range(n + warmup):
        t0 = time.perf_counter()
        sess.run(None, {inp_name: dummy})
        if i >= warmup:
            latencies.append((time.perf_counter() - t0) * 1000)
    return latencies


results = {}
for label, path in [("fp32", "artifacts/model.onnx"), ("int8", "artifacts/model_int8.onnx")]:
    if Path(path).exists():
        print(f"Benchmarking {label}...")
        results[label] = benchmark(path)
        print(
            f"  mean={np.mean(results[label]):.2f} ms  p95={np.percentile(results[label], 95):.2f} ms"
        )

## Summary Table

In [ ]:
rows = []
for label, lats in results.items():
    rows.append(
        {
            "format": label,
            "mean_ms": round(np.mean(lats), 2),
            "median_ms": round(np.median(lats), 2),
            "p95_ms": round(np.percentile(lats, 95), 2),
            "p99_ms": round(np.percentile(lats, 99), 2),
            "std_ms": round(np.std(lats), 2),
            "throughput_rps": round(1000 / np.mean(lats), 1),
        }
    )
pd.DataFrame(rows)

## Latency Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for label, lats in results.items():
    ax.plot(lats, label=label, alpha=0.7)
ax.set_xlabel("Run index")
ax.set_ylabel("Latency (ms)")
ax.set_title("Inference latency over time")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label, lats in results.items():
    ax.hist(lats, bins=30, alpha=0.6, label=label)
ax.set_xlabel("Latency (ms)")
ax.set_title("Latency histogram")
ax.legend()
plt.tight_layout()
plt.show()